In [1]:
from utils import *

# Week 2
- collect enceladus seed set
- make a csv table with compound name, kegg ID, kegg name, comment, source

In [ ]:
# check if cpd is used in any reactions

for r, cpds in rn2cpds.items():
    for c in cpds:
        if c == 'C15521':
            print(r, cpds)

In [ ]:
# read enceladus_seed_cpds
df = pd.read_csv('../enceladus_assets/enceladus_seed_cpds.csv')
df

In [ ]:
enceladus_seed = set(df['KEGG_ID'])
enceladus_seed.remove('0')
print(len(enceladus_seed))

In [ ]:
# divide seeds by paper
papers = {}
papers['peter'] = df[df['source'].str.startswith('Peter')]
papers['parker'] = df[df['source'].str.startswith('Parker')]
papers['yoshimura'] = df[df['source'].str.startswith('Yoshimura')]
papers['takano'] = df[df['source'].str.startswith('Takano')]
papers['oba'] = df[df['source'].str.startswith('Oba')]

In [ ]:
# draw them

import requests
from rdkit import Chem
from rdkit.Chem import Draw
from IPython.display import display
from PIL import Image

def get_kegg_compound_smiles(kegg_id):
    url = f'http://rest.kegg.jp/get/{kegg_id}/mol'
    response = requests.get(url)
    if response.status_code == 200:
        mol_data = response.text
        mol = Chem.MolFromMolBlock(mol_data)
        return mol
    else:
        raise ValueError(f"Error fetching data for {kegg_id}: {cpd2name.get(kegg_id, '')}")

def draw_multiple_molecules(molecules, mol_labels=None):
    img = Draw.MolsToGridImage(molecules, molsPerRow=10, subImgSize=(400, 400), legends=mol_labels, maxMols=200)
    display(img)

    with open('cpds.png', mode='wb') as f:
        f.write(img.data)
        
def drawMols(molecule_kegg_ids):
    molecules = []
    labels = []

    # from SMILES
    for kegg_id in molecule_kegg_ids:
        try:
            mol_kegg = get_kegg_compound_smiles(kegg_id)
            if mol_kegg:
                molecules.append(mol_kegg)
                labels.append(cpd2name[kegg_id])
        except ValueError as e:
            print(e)
            
    if molecules:
        draw_multiple_molecules(molecules, labels)

In [ ]:
# drawMols(enceladus_seed)

In [ ]:
result = pd.read_pickle(f'../data/fold_results/runs_fastest_slowest/2024-07-24_11-10-36_no_lookahead_preExpansion_NONE_83276.pkl.gz')

In [ ]:
seed80 = [c for c, i in result.cpds_folditer.items() if i == 0]
len(seed80)

In [ ]:
aa_cids = set(["C00037",
    "C00041",
    "C00065",
    "C00188",
    "C00183",
    "C00407",
    "C00123",
    "C00148",
    "C00049",
    "C00025"])

seed70 = set(seed80) - aa_cids
len(seed70)

In [ ]:
# drawMols(seed70)

## Seed set comparison

In [ ]:
from matplotlib_venn import venn2

# Create a Venn diagram for 2 sets
venn2(subsets=[set(seed70), enceladus_seed], set_labels=('seed70', 'enceladus'),
      set_colors=('red', 'skyblue'), alpha=0.7)

# plt.savefig('venn_ToTAL.svg', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
for c in seed70 & enceladus_seed:
    print(c, cpd2name[c])

In [ ]:
# drawMols(seed70 - enceladus_seed)

In [ ]:
# metals + generic cpds
metals = ['Z00020', 'Z00002', 'C00028', 'Z00030', 'C00205', 'Z00006', 'Z00015', 'C00050', 'Z00053', 'Z00055', 'Z00063', 'C17023', 'C00030', 'Z00069', 'Z00064', 'Z00001', 'Z00054', 'Z00060', 'Z00070', 'Z00029', 'Z00034', 'C22155', 'Z00067', 'Z00062', 'Z00033', 'C00023', 'C01330', 'C00071', 'C00034', 'C19609', 'C14819', 'C01335', 'C00069', 'C00305', 'C14818', 'C00070', 'C00175', 'C00038', 'C00238', 'C00080', 'C00150']
for m in metals:
    print(m, cpd2name[m])
print(len(metals))

## Run network expansion

In [ ]:
# function to draw trajectory

def drawNE(fm):
    iter2cpds = {}
    for c, i in fm.scope.cpd_iteration_dict.items():
        if i not in iter2cpds.keys():
            iter2cpds[i] = [c]
        else:
            iter2cpds[i].append(c)
            
    iter2cpdsNum = {}
    for i in iter2cpds.keys():
        iter2cpdsNum[i] = len(iter2cpds[i])
        
    plt.figure(figsize=(30, 5))
    plt.plot(iter2cpdsNum.values(), color='k')
    
    plt.xlim([0, max(iter2cpdsNum.keys())])
    plt.ylim([0, max(iter2cpdsNum.values())+10])
    plt.xlabel('iteration', fontsize='18')
    plt.ylabel('# cpds discovered', fontsize='18')
    plt.xticks(fontsize='18')
    plt.yticks(fontsize='18')
    
    # plt.savefig('vanilla_iter_vs_cpds_10AA.svg', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
import networkExpansionPy.folds as nf
import pandas as pd
from collections import Counter
from pathlib import PurePath, Path

# seed = sys.argv[1]
# random.seed(seed)
asset_path = nf.asset_path

# for vanilla
METABOLISM_PATH = PurePath(asset_path, "metabolic_networks","metabolism.v8.01May2023.pkl") # path to metabolism object pickle
RN2RULES_PATH = PurePath(asset_path,"rn2fold","rn2rules.20230224.pkl") # path to rn2rules object pickle
SEED_CPDS_PATH = PurePath(asset_path, "compounds", "seeds.Goldford2022.csv") # path to seed compounds csv

# for FOLD-GATED
ALGORITHM = "no_look_ahead_rules"
WRITE = True # write result to disk
WRITE_TMP = False # write after each iteration
CUSTOM_WRITE_PATH = None # if writing result, custom path to write to
STR_TO_APPEND_TO_FNAME = None # if writing result, str to append to filename
ORDERED_OUTCOME = False # ignore random seed and always choose folds based on sort order
IGNORE_REACTION_VERSIONS = True # when maximizing for reactions, don't count versioned reactions

## Metabolism
metabolism = pd.read_pickle(METABOLISM_PATH)
# remove reactions that produce H2O2 before O2
H2O2_rns = ['R00017_v1', 'R03532_v1', 'R09507_v1', 'R09740_v1', 'R09741_v1', 'R11522', 'R12455', 'R12454']
condition = ((metabolism.network['rn'].isin(H2O2_rns)) & (metabolism.network['direction'] == 'reverse'))
metabolism.network = metabolism.network[~condition]
assert 'R00017_v1' not in list(metabolism.network['rn'])

## FoldRules
rn2rules = pd.read_pickle(RN2RULES_PATH)
foldrules = nf.FoldRules.from_rn2rules(rn2rules)

## Modify seeds with AA and GATP_rns
aa_cids = set(["C00037",
    "C00041",
    "C00065",
    "C00188",
    "C00183",
    "C00407",
    "C00123",
    "C00148",
    "C00049",
    "C00025"])

GATP_rns = {'R00200_gATP_v1',
    'R00200_gATP_v2',
    'R00430_gGTP_v1',
    'R00430_gGTP_v2',
    'R01523_gATP_v1',
    'R04144_gATP_v1',
    'R04208_gATP',
    'R04463_gATP',
    'R04591_gATP_v1',
    'R06836_gATP',
    'R06974_gATP',
    'R06975_gATP_v1'}

# seed = nf.Params(
#     rns = set(metabolism.network["rn"]) - set(rn2rules) | GATP_rns,  # start with non-fold-gated reactions
#     cpds = set((pd.read_csv(SEED_CPDS_PATH)["ID"])),
#     folds = set(['spontaneous'])
# )

## modified seed (seed70, enceladus_seed etc.)
seed = nf.Params(
    rns = set(metabolism.network["rn"]) | GATP_rns,  # start with non-fold-gated reactions + GATP_rns
    cpds = enceladus_seed | set(['C00009']) | set(metals),
    folds = set(['spontaneous'])
)

## Inititalize fold metabolism
fm = nf.FoldMetabolism(metabolism, foldrules, seed)

In [ ]:
full = fm.scope.cpds

In [ ]:
len(fm.scope.cpds)

In [ ]:
for c in full - fm.scope.cpds:
    print(c, cpd2name[c])

In [ ]:
# seed70
drawNE(fm)

In [ ]:
# enceladus_seed
drawNE(fm)
len(fm.seed.cpds), len(fm.scope.cpds)

In [ ]:
# enceladus_seed + metals
drawNE(fm)
len(fm.seed.cpds), len(fm.scope.cpds)

In [ ]:
iter2cpds = {}
for c, i in fm.scope.cpd_iteration_dict.items():
    if i not in iter2cpds.keys():
        iter2cpds[i] = [c]
    else:
        iter2cpds[i].append(c)

In [ ]:
# print cpds at iter 
for c in iter2cpds[1]:
    print(c, cpd2name[c])

In [ ]:
venn2(subsets=[set(seed70), (enceladus_seed | set(metals))], set_labels=('seed70', 'enceladus+metal'),
      set_colors=('red', 'skyblue'), alpha=0.7)

# plt.savefig('venn_ToTAL.svg', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# which seed is necessary to "rescue"?

rescue = set(seed70) - (enceladus_seed | set(metals))
for c in rescue:
    print(c, cpd2name[c])

In [ ]:
# HCO3-?, Molybdate?, Tungstate?, Hydrogen selenide?, Formate? Acetate?

In [ ]:
# # add 1 seed at a time
# plusOne2ns = {}

# for plusOne in rescue:
    
#     ## FoldRules
#     rn2rules = pd.read_pickle(RN2RULES_PATH)
#     foldrules = nf.FoldRules.from_rn2rules(rn2rules)
    
#     seed = nf.Params(
#         rns = set(metabolism.network["rn"]) | GATP_rns,  # start with non-fold-gated reactions + GATP_rns
#         cpds = enceladus_seed | set(metals) | set([plusOne]),
#         folds = set(['spontaneous'])
#     )
    
#     fm = nf.FoldMetabolism(metabolism, foldrules, seed)
#     print(len(fm.scope.cpds), plusOne, cpd2name[plusOne])
#     plusOne2ns[plusOne] = len(fm.scope.cpds)

In [ ]:
# dict2csv(plusOne2ns, '../../plusOne2ns.csv')
plusOne2ns = csv2dict('../enceladus_assets/plusOne2ns.csv')

In [ ]:
for key in plusOne2ns.keys():
    print(plusOne2ns[key], key, cpd2name[key])

In [ ]:
# # add 2 seeds at a time
# import itertools

# plusTwo2ns = {}

# for combo in itertools.combinations(rescue, 2):
#     seed = nf.Params(
#         rns = set(metabolism.network["rn"]) | GATP_rns,  # start with non-fold-gated reactions + GATP_rns
#         cpds = enceladus_seed | set(metals) | set(combo),
#         folds = set(['spontaneous'])
#     )
    
#     ## Inititalize fold metabolism
#     fm = nf.FoldMetabolism(metabolism, foldrules, seed)
#     print(len(fm.scope.cpds), combo, cpd2name[combo[0]], cpd2name[combo[1]])
#     plusTwo2ns[combo] = len(fm.scope.cpds)

In [ ]:
# for key in plusTwo2ns.keys():
#     print(plusTwo2ns[key], key, cpd2name[key[0]], cpd2name[key[1]])

In [ ]:
# enceladus with phosphate, no 'metals'

seed = nf.Params(
        rns = set(metabolism.network["rn"]) | GATP_rns,  # start with non-fold-gated reactions + GATP_rns
        cpds = enceladus_seed | set(['C00009']),
        folds = set(['spontaneous'])
    )
    
## Inititalize fold metabolism
fm = nf.FoldMetabolism(metabolism, foldrules, seed)
print(len(fm.scope.cpds))

In [ ]:
# # do paper-by-paper
# paper2ns = {}

# for paper, cpds_df in papers.items():
#     s = set((cpds_df['KEGG_ID']))
#     s.remove('0')
    
#     seed = nf.Params(
#             rns = set(metabolism.network["rn"]) | GATP_rns,  # start with non-fold-gated reactions + GATP_rns
#             cpds = s,
#             folds = set(['spontaneous'])
#         )
    
#     ## Inititalize fold metabolism
#     fm = nf.FoldMetabolism(metabolism, foldrules, seed)
#     print(len(s), len(fm.scope.cpds), paper)
#     paper2ns[paper] = len(fm.scope.cpds)

In [ ]:
# dict2csv(paper2ns, '../../paper2ns.csv')
paper2ns = csv2dict('../enceladus_assets/paper2ns.csv')

## Plots

In [ ]:
# 22 29 peter
# 33 55 parker
# 10 11 yoshimura
# 36 53 takano
# 7 5 oba

In [ ]:
# Create a horizontal bar plot
plt.figure(figsize=(6, 6))

keys, values = list(plusOne2ns.keys()), list(plusOne2ns.values())
keys.append('enceladus_seed + metals')
values.append(254)
keys.append('enceladus_seed + orthophosphate')
values.append(156)
keys.append('enceladus_seed')
values.append(150)
keys.extend(paper2ns.keys())
values.extend(paper2ns.values())
keys.append('full network')
values.append(4294)

plt.barh(keys, values, color='skyblue')
for index, value in enumerate(values):
    plt.text(value + 0.5, index, str(value), va='center', ha='left', color='black')

# red for seed size
values = [145+1]*18
values.extend([145, 106, 107, 22, 33, 10, 36, 7, 70])
plt.barh(keys, values, color='red')


plt.xlabel('Network size')
plt.show()

# Week 3-4

## Add more cpds to "metals" based on what's available (according to Yasu)
These are also essential for WL pathway!

In [ ]:
for c in ['C00009', 'C20679', 'C01528', 'C06232']:
    print(c, cpd2name[c])

metals.extend(['C00009', 'C20679', 'C01528', 'C06232'])
len(metals)

## 1) Is the Enceladus expansion trajectory significantly different from vanilla?
Calculate the Spearman rank corr. coef. of [vanilla+10AA vs. Enceladus] cpd discovery order 

In [ ]:
print(len(enceladus_seed | set(metals)), len(seed80))

In [ ]:
## enceladus
seed = nf.Params(
        rns = set(metabolism.network["rn"]) | GATP_rns,  # start with non-fold-gated reactions + GATP_rns
        cpds = enceladus_seed | set(metals),
        folds = set(['spontaneous'])
    )
    
## Inititalize fold metabolism
fm_e = nf.FoldMetabolism(metabolism, foldrules, seed)
print(len(fm_e.scope.cpds))

## vanilla + 10AA
seed = nf.Params(
    rns = set(metabolism.network["rn"]) | GATP_rns,  # start with non-fold-gated reactions
    cpds = set((pd.read_csv(SEED_CPDS_PATH)["ID"])) | aa_cids,
    folds = set(['spontaneous'])
)
    
## Inititalize fold metabolism
fm = nf.FoldMetabolism(metabolism, foldrules, seed)
print(len(fm.scope.cpds))

In [ ]:
sp, p = spearman(fm.scope.cpd_iteration_dict, fm_e.scope.cpd_iteration_dict)
print(f'Spearman rank corr. = {sp}, p-value = {p}')
scatter(fm.scope.cpd_iteration_dict, fm_e.scope.cpd_iteration_dict, 'vanilla', 'enceladus')

### This has too many data points! Let's look at the level of KEGG pathways (modules).
For each KEGG metabolic module, record the "number of iterations until completion"

In [ ]:
# function to check timing of pathway completion 
def pathwayCheck(fm, mods, output=True):
    mod2timing = {}
    iter2rn = {}
    mods_copy = copy.deepcopy(mods)
    
    for rn, i in fm.scope.rn_iteration_dict.items():
        if i not in iter2rn.keys():
            iter2rn[i] = [rn]
        else:
            iter2rn[i].append(rn)
    
    rn_list = []
    for i, rns in iter2rn.items():
        # keep updating rn_list
        rn_list.extend(rns)
        
        # for each mods module, check if all rns in module are in rn_list
        for m in mods_copy:
            if all(item in [item[:6] for item in rn_list] for item in module2rns[m]):
                if output:
                    print(f'{m}: {module2name[m]} is feasible at {i}')
                mod2timing[m] = i
                mods_copy.remove(m)
    return mod2timing

In [ ]:
mod2t = pathwayCheck(fm, list(module2name.keys()), output=False)

In [ ]:
mod2t_e = pathwayCheck(fm_e, list(module2name.keys()), output=False)

In [ ]:
scatter(mod2t, mod2t_e, 'vanilla+10AA', 'enceladus')

In [ ]:
# # bokeh plot:
# d1 = mod2t
# d2 = mod2t_e

# output_file("scatter.html")
# p = figure(width=800, height=800, title="iteration at module completion: vanilla+10AA vs. Enceladus")

# # Add annotations
# valid_keys, data1, data2 = todata(d1, d2)

# source = {'x': data1, 'y': data2, 'label': [key + ' ' + module2name[key] for key in valid_keys]}
# p.scatter('x', 'y', source=source, size=10, color='blue', alpha=0.5)

# # Add hover tool
# hover = HoverTool()
# hover.tooltips = [("", "@label"), ("x", "@x"), ("y", "@y")]
# p.add_tools(hover)

# # Customize plot
# p.line([0, max(data1)], [0, max(data1)], line_width=2, color='black', alpha=0.7)
# p.xaxis.axis_label = 'vanilla+10AA'
# p.yaxis.axis_label = 'enceladus_seed'
# p.xaxis.ticker = [0, 20, 40, 60, 80]  # Convert range to list
# p.xgrid.grid_line_color = None

# # Show the plot
# show(p)

## 1b) Which compound is important for expansion? 
>i.e. Can we change the expansion speed by adding another compound to the enceladus_seed?

Add one cpd at a time (from ~4,000 cpds of full network) => check effect on expansion

In [ ]:
# takes 3 hours for ~4,000 cpds

# cpd2effect = {}

# for plusOne in fm.scope.cpds - (set(enceladus_seed) | set(['C00009']) | set(metals)):
#     seed = nf.Params(
#             rns = set(metabolism.network["rn"]) | GATP_rns,  # start with non-fold-gated reactions + GATP_rns
#             cpds = set(enceladus_seed) | set(metals) | set([plusOne]),
#             folds = set(['spontaneous'])
#         )
        
#     fm_p = nf.FoldMetabolism(metabolism, foldrules, seed)
#     sp, p = spearman(fm_p.scope.cpd_iteration_dict, fm_e.scope.cpd_iteration_dict)
#     cpd2effect[plusOne] = sp

In [ ]:
# dict2csv(cpd2effect, '../../cpd2effect_enceladus+1.csv')
cpd2effect = csv2dict('../enceladus_assets/cpd2effect_enceladus+1.csv')

In [ ]:
histogram(cpd2effect, bins=100, x_axis='Spearman rank corr. coef.',  y_axis='count')

print cpds with sp < 0.90

In [ ]:
for c, sp in cpd2effect.items():
    if sp < 0.90:
        print(c, sp, cpd2name[c])

In [ ]:
# for sp < 0.9, draw scatterplot => identify major patterns
# they all look similar

# for c, sp in cpd2effect.items():
#     if sp < 0.90:
#         print(c, sp, cpd2name[c])

#         # run network expansion again with plusOne
#         seed = nf.Params(
#         rns = set(metabolism.network["rn"]) | GATP_rns, 
#         cpds = set(enceladus_seed) | set(metals) | set([c]),
#         folds = set(['spontaneous']))
        
#         fm_p = nf.FoldMetabolism(metabolism, foldrules, seed)

#         # get module discovery timing
#         mod2t_p = pathwayCheck(fm_p, list(module2name.keys()))

#         # draw scatterplot
#         scatter(mod2t_e, mod2t_p, 'enceladus', 'enceladus +1')

In [ ]:
# # bokeh for +C00154

# # run network expansion again
# c = 'C00154'
# seed = nf.Params(
# rns = set(metabolism.network["rn"]) | GATP_rns, 
# cpds = set(enceladus_seed) | set(metals) | set([c]),
# folds = set(['spontaneous']))
# fm_p = nf.FoldMetabolism(metabolism, foldrules, seed)

# # get module discovery timing
# mod2t_p = pathwayCheck(fm_p, list(module2name.keys()), output=False)

# # bokeh plot:
# d1 = mod2t_e
# d2 = mod2t_p

# output_file(f"scatter_{c}.html")
# p = figure(width=800, height=800, title=f"iteration at module completion: Enceladus vs. Enceladus + {c}: {cpd2name[c]}")

# # Add annotations
# valid_keys, data1, data2 = todata(d1, d2)

# source = {'x': data1, 'y': data2, 'label': [key + ' ' + module2name[key] for key in valid_keys]}
# p.scatter('x', 'y', source=source, size=10, color='blue', alpha=0.5)

# # Add hover tool
# hover = HoverTool()
# hover.tooltips = [("", "@label"), ("x", "@x"), ("y", "@y")]
# p.add_tools(hover)

# # Customize plot
# p.line([0, max(data1)], [0, max(data1)], line_width=2, color='black', alpha=0.7)
# p.xaxis.axis_label = 'enceladus_seed'
# p.yaxis.axis_label = f'enceladus_seed +{c}'
# p.xaxis.ticker = [0, 20, 40, 60, 80]  # Convert range to list
# p.xgrid.grid_line_color = None

# # Show the plot
# show(p)

relax threshold to .95

In [ ]:
for c, sp in cpd2effect.items():
    if sp < 0.95:
        print(c, sp, cpd2name[c])

In [ ]:
check = ['C00016', 'C15672', 'C22349', 'C00101', 'C00251']
for c in check:
    print(c, cpd2name[c])

In [ ]:
# # bokeh for 5 compounds; takes time! => saved as html
# for c in check:
#     # run network expansion again
#     seed = nf.Params(
#     rns = set(metabolism.network["rn"]) | GATP_rns, 
#     cpds = set(enceladus_seed) | set(metals) | set([c]),
#     folds = set(['spontaneous']))
#     fm_p = nf.FoldMetabolism(metabolism, foldrules, seed)
    
#     # get module discovery timing
#     mod2t_p = pathwayCheck(fm_p, list(module2name.keys()), output=False)
    
#     # bokeh plot:
#     d1 = mod2t_e
#     d2 = mod2t_p
    
#     output_file(f"scatter_{c}.html")
#     p = figure(width=800, height=800, title=f"iteration at module completion: Enceladus vs. Enceladus + {c}: {cpd2name[c]}")
    
#     # Add annotations
#     valid_keys, data1, data2 = todata(d1, d2)
    
#     source = {'x': data1, 'y': data2, 'label': [key + ' ' + module2name[key] for key in valid_keys]}
#     p.scatter('x', 'y', source=source, size=10, color='blue', alpha=0.5)
    
#     # Add hover tool
#     hover = HoverTool()
#     hover.tooltips = [("", "@label"), ("x", "@x"), ("y", "@y")]
#     p.add_tools(hover)
    
#     # Customize plot
#     p.line([0, max(data1)], [0, max(data1)], line_width=2, color='black', alpha=0.7)
#     p.xaxis.axis_label = 'enceladus_seed'
#     p.yaxis.axis_label = f'enceladus_seed +{c}'
#     p.xaxis.ticker = [0, 20, 40, 60, 80]  # Convert range to list
#     p.xgrid.grid_line_color = None
    
#     # Show the plot
#     show(p)

## 2) Leave-One-Out analysis
If we remove one seed cpd at a time, which cpd will have the largest reduction in network size?

### with vanilla expansion

In [ ]:
# cpd2loo_vanilla = {}

# for c in set((pd.read_csv(SEED_CPDS_PATH)["ID"])):
#     seed = nf.Params(
#             rns = set(metabolism.network["rn"]) | GATP_rns,  # start with non-fold-gated reactions + GATP_rns
#             cpds = set((pd.read_csv(SEED_CPDS_PATH)["ID"])) - set([c]),
#             folds = set(['spontaneous'])
#         )
        
#     fm_loo = nf.FoldMetabolism(metabolism, foldrules, seed)
#     cpd2loo_vanilla[c] = len(fm_loo.scope.cpds)

In [ ]:
# dict2csv(cpd2loo_vanilla, '../../cpd2loo_vanilla.csv')
cpd2loo_vanilla = csv2dict('../enceladus_assets/cpd2loo_vanilla.csv')

In [ ]:
for c, i in cpd2loo_vanilla.items():
    print(i, c, cpd2name[c])

### with enceladus seedset

In [ ]:
len(enceladus_seed | set(metals))

In [ ]:
# cpd2loo = {}

# for c in enceladus_seed | set(metals):
#     seed = nf.Params(
#             rns = set(metabolism.network["rn"]) | GATP_rns,  # start with non-fold-gated reactions + GATP_rns
#             cpds = (enceladus_seed | set(metals)) - set([c]),
#             folds = set(['spontaneous'])
#         )
        
#     fm_loo = nf.FoldMetabolism(metabolism, foldrules, seed)
#     cpd2loo[c] = len(fm_loo.scope.cpds)

In [ ]:
# dict2csv(cpd2loo, '../../cpd2loo.csv')
cpd2loo = csv2dict('../enceladus_assets/cpd2loo.csv')

In [ ]:
for c, i in cpd2loo.items():
    print(i, c, cpd2name[c])

### Leave one-group out
enceladus_seed set is robust to perturbation... let's remove groups of cpds

In [ ]:
for paper in papers:
    s = set(papers[paper]['KEGG_ID'])
    s.remove('0')
    print(paper, s)

In [ ]:
# cpd2loo_paper = {}

# for paper in papers:
#     s = set(papers[paper]['KEGG_ID'])
#     s.remove('0')
    
#     seed = nf.Params(
#             rns = set(metabolism.network["rn"]) | GATP_rns,  # start with non-fold-gated reactions + GATP_rns
#             cpds = (enceladus_seed | set(metals)) - s,
#             folds = set(['spontaneous'])
#         )
        
#     print(len((enceladus_seed | set(metals)) - s))
#     fm_loo = nf.FoldMetabolism(metabolism, foldrules, seed)
#     cpd2loo_paper[paper] = len(fm_loo.scope.cpds)

In [ ]:
# dict2csv(cpd2loo_paper, '../../cpd2loo_paper.csv')
cpd2loo_paper = csv2dict('../enceladus_assets/cpd2loo_paper.csv')

In [ ]:
cpd2loo_paper
# removing parker or yoshimura restricts expansion

### Which amino acid is important?
Let's remove all 'parker' amino acids => add back one AA at a time

In [ ]:
parker = set(papers['parker']['KEGG_ID'])
parker.remove('0')

In [ ]:
len((enceladus_seed | set(metals)))

In [ ]:
len((enceladus_seed | set(metals)) - parker)

In [ ]:
# plusOne2ns_AA = {}

# for AA in parker:
    
#     seed = nf.Params(
#             rns = set(metabolism.network["rn"]) | GATP_rns,  # start with non-fold-gated reactions + GATP_rns
#             cpds = ((enceladus_seed | set(metals)) - parker) | set([AA]),
#             folds = set(['spontaneous'])
#         )
        
#     fm_loo = nf.FoldMetabolism(metabolism, foldrules, seed)
#     print(len(fm_loo.seed.cpds))
#     plusOne2ns_AA[AA] = len(fm_loo.scope.cpds)

In [ ]:
# dict2csv(plusOne2ns_AA, '../../plusOne2ns_AA.csv')
plusOne2ns_AA = csv2dict('../enceladus_assets/plusOne2ns_AA.csv')
plusOne2ns_AA.pop('0')

In [ ]:
for c, ns in plusOne2ns_AA.items():
    print(ns, c, cpd2name[c])

### Which compounds are important?
- orthophosphate
- sulfur source
- Glutamine!

## 3) pH=9

In [ ]:
asset_path = nf.asset_path

# for vanilla
METABOLISM_PATH = PurePath(asset_path, "metabolic_networks","metabolism.v8.02Sep2024_pH9.pkl") # path to metabolism object pickle
RN2RULES_PATH = PurePath(asset_path,"rn2fold","rn2rules.20230224.pkl") # path to rn2rules object pickle
SEED_CPDS_PATH = PurePath(asset_path, "compounds", "seeds.Goldford2022.csv") # path to seed compounds csv

# for FOLD-GATED
ALGORITHM = "no_look_ahead_rules"
WRITE = True # write result to disk
WRITE_TMP = False # write after each iteration
CUSTOM_WRITE_PATH = None # if writing result, custom path to write to
STR_TO_APPEND_TO_FNAME = None # if writing result, str to append to filename
ORDERED_OUTCOME = False # ignore random seed and always choose folds based on sort order
IGNORE_REACTION_VERSIONS = True # when maximizing for reactions, don't count versioned reactions

## Metabolism
metabolism = pd.read_pickle(METABOLISM_PATH)
# remove reactions that produce H2O2 before O2
H2O2_rns = ['R00017_v1', 'R03532_v1', 'R09507_v1', 'R09740_v1', 'R09741_v1', 'R11522', 'R12455', 'R12454']
condition = ((metabolism.network['rn'].isin(H2O2_rns)) & (metabolism.network['direction'] == 'reverse'))
metabolism.network = metabolism.network[~condition]
assert 'R00017_v1' not in list(metabolism.network['rn'])

## FoldRules
rn2rules = pd.read_pickle(RN2RULES_PATH)
foldrules = nf.FoldRules.from_rn2rules(rn2rules)

GATP_rns = {'R00200_gATP_v1',
    'R00200_gATP_v2',
    'R00430_gGTP_v1',
    'R00430_gGTP_v2',
    'R01523_gATP_v1',
    'R04144_gATP_v1',
    'R04208_gATP',
    'R04463_gATP',
    'R04591_gATP_v1',
    'R06836_gATP',
    'R06974_gATP',
    'R06975_gATP_v1'}

## modified seed (seed70, enceladus_seed etc.)
seed = nf.Params(
    rns = set(metabolism.network["rn"]) | GATP_rns,  # start with non-fold-gated reactions + GATP_rns
    cpds = enceladus_seed | set(metals),
    folds = set(['spontaneous'])
)

## Inititalize fold metabolism
fm_9 = nf.FoldMetabolism(metabolism, foldrules, seed)

In [ ]:
len(fm_9.scope.cpds), len(fm_e.scope.cpds)

In [ ]:
len(fm_e.scope.cpds) - len(fm_9.scope.cpds)

In [ ]:
for c in fm_e.scope.cpds - fm_9.scope.cpds:
    print(fm_e.scope.cpd_iteration_dict[c], c, cpd2name.get(c, 'no name'))

## 4) takeaways? future plans?